# Create mastersheet from
- app/data/tycho_cases.csv
- app/data/nis_vacc_coverage.csv
- app/data/missing_cases.csv
- app/data/population.csv

In [3]:
import pandas as pd

# -------------------------
# Load data
# -------------------------

cases_df = pd.read_csv(
    "../app/data/tycho_cases.csv"
)

missing_cases_df = pd.read_csv(
    "../app/data/missing_cases.csv"
)

population_df = pd.read_csv(
    "../app/data/population.csv"
)

coverage_df = pd.read_csv(
    "../app/data/nis_vacc_coverage.csv"
)


# -------------------------
# Combine Tycho + MMWR cases
# -------------------------

cases_df = pd.merge(
    cases_df,
    missing_cases_df,
    on=["year", "state"],
    how="outer",
    suffixes=("_tycho", "_mmwr")
)


# Prefer Tycho values when available.
# Use MMWR to fill missing Tycho values.
cases_df["measles_cases"] = (
    cases_df["measles_cases_tycho"]
    .combine_first(
        cases_df["measles_cases_mmwr"]
    )
)

cases_df["mumps_cases"] = (
    cases_df["mumps_cases_tycho"]
    .combine_first(
        cases_df["mumps_cases_mmwr"]
    )
)


# Keep final case columns
cases_df = cases_df[
    [
        "year",
        "state",
        "measles_cases",
        "mumps_cases",
        "pertussis_cases"
    ]
]


# Sort
cases_df = cases_df.sort_values(
    ["state", "year"]
).reset_index(drop=True)


# -------------------------
# Merge datasets
# -------------------------

yearly_coverage_cases_df = pd.merge(
    cases_df,
    coverage_df,
    on=["year", "state"]
)

yearly_coverage_cases_df = pd.merge(
    yearly_coverage_cases_df,
    population_df,
    on=["year", "state"],
    how="left"
)


# -------------------------
# Calculate cases per 100K
# -------------------------

diseases = [
    "measles",
    "mumps",
    "pertussis"
]

for disease in diseases:

    yearly_coverage_cases_df[
        f"{disease}_cases_per_100k"
    ] = (
        yearly_coverage_cases_df[f"{disease}_cases"]
        / yearly_coverage_cases_df["population"]
        * 100_000
    )

In [4]:
yearly_coverage_cases_df.head()

,year,state,measles_cases,mumps_cases,pertussis_cases,measles_coverage_pct,measles_n,measles_n_vaccinated,mumps_coverage_pct,mumps_n,mumps_n_vaccinated,pertussis_coverage_pct,pertussis_n,pertussis_n_vaccinated,population,measles_cases_per_100k,mumps_cases_per_100k,pertussis_cases_per_100k
0,1995,AK,NaN,13.0,1.0,89.86,194.0,177.0,89.86,194.0,177.0,77.58,194.0,159.0,604412.0,NaN,2.150851,0.165450
1,1996,AK,63.0,3.0,4.0,84.55,260.0,225.0,84.55,260.0,225.0,78.25,260.0,210.0,608569.0,10.352154,0.492960,0.657280
2,1997,AK,48.0,4.0,14.0,87.41,291.0,257.0,87.41,291.0,257.0,80.98,291.0,242.0,612968.0,7.830751,0.652563,2.283969
3,1998,AK,29.0,2.0,15.0,87.06,34.0,30.0,87.06,34.0,30.0,82.01,34.0,28.0,619932.0,4.677932,0.322616,2.419620
4,1999,AK,29.0,3.0,5.0,90.67,349.0,321.0,90.67,349.0,321.0,83.54,349.0,296.0,624779.0,4.641641,0.480170,0.800283


In [6]:
yearly_coverage_cases_df.to_csv("../app/data/cases_coverage_ms.csv", index=False)